In [1]:
import pandas as pd
import re

In [2]:
df = pd.read_csv("data/raw/hsk_vocabulary_raw_uncleaned.csv")
df

,hsk_level,no_in_level,word,pinyin,part_of_speech,english
0,1,1,爱,ài,verb,to love
1,1,2,⼋,bā,number,eight
2,1,3,爸爸,bàba,noun,dad
3,1,4,吧,ba,auxiliary,"particle indicating suggestion, confirmation, ..."
4,1,5,⽩天,báitiān,noun,daytime
...,...,...,...,...,...,...
10995,7-Sep,5596,作息,zuòxī,动,NaN
10996,7-Sep,5597,座右铭,zuòyòumíng,名,NaN
10997,7-Sep,5598,作战,zuòzhàn,动,NaN
10998,7-Sep,5599,做证,zuòzhèng,动,NaN


In [3]:
df['part_of_speech'].unique()

<ArrowStringArray>
[            'verb',           'number',             'noun',
        'auxiliary', 'number、 （adverb）',       'classifier',
      'noun、suffix',        'noun、verb',           'adverb',
                nan,
 ...
             '名、 形',              '副、连',              '代、副',
             '拟声、动',             '名、后缀',            '动、形、副',
             '数量、名',              '连、副',                '数',
              '动、数']
Length: 188, dtype: str

In [4]:
pos = {
    '动': 'verb',
    '名': 'noun',
    '形': 'adjective',
    '副': 'adverb',
    '量': 'classifier',
    '连': 'conjunction',
    '介': 'preposition',
    '代': 'pronoun',
    '数': 'number',
    '助': 'auxiliary',
    '叹': 'interjection',
    '拟声': 'onomatopoeia',
    '前缀': 'prefix',
    '后缀': 'suffix',
    '数量': 'number|classifier'
}

In [5]:
def clean_pos(raw_val):
    if pd.isna(raw_val):
        return raw_val
    val = str(raw_val).strip()
    val = val.replace('numberclassifier', 'number|classifier')
    tokens = re.findall(r"[\u4e00-\u9fff]+|[a-zA-Z]+", val)
    cleaned_tokens = []
    for token in tokens:
        translated = pos.get(token, token)
        for part in translated.split("|"):
            if part not in cleaned_tokens:
                cleaned_tokens.append(part)
    return "|".join(cleaned_tokens)

df['part_of_speech'] = df['part_of_speech'].apply(clean_pos)

In [6]:
df['part_of_speech'].unique()

<ArrowStringArray>
[                    'verb',                   'number',
                     'noun',                'auxiliary',
            'number|adverb',               'classifier',
              'noun|suffix',                'noun|verb',
                   'adverb',                        nan,
 ...
     'noun|classifier|verb',    'noun|adverb|adjective',
             'onomatopoeia',    'conjunction|auxiliary',
 'adverb|number|classifier',         'verb|noun|adverb',
       'verb|adverb|prefix',    'verb|adverb|adjective',
        'onomatopoeia|verb',              'verb|number']
Length: 109, dtype: str

In [7]:
df[df['part_of_speech'].str.contains(r'\|', na=False)]

,hsk_level,no_in_level,word,pinyin,part_of_speech,english
6,1,7,半,bàn,number|adverb,"half; (half, semi-)"
10,1,11,边,biān,noun|suffix,side; suffix indicating direction
11,1,12,病,bìng,noun|verb,"illness, disease; to be ill"
32,1,33,点,diǎn,classifier|noun,"measure word for small round things; (point, dot)"
43,1,44,对,duì,adjective|preposition|verb|classifier,"correct, right; to, toward, for; (to treat; pair)"
...,...,...,...,...,...,...
10876,7-Sep,5477,坠,zhuì,verb|noun,NaN
10896,7-Sep,5497,滋润,zīrùn,adjective|verb,NaN
10909,7-Sep,5510,⾃称,zìchēng,verb|noun,NaN
10946,7-Sep,5547,纵横,zònghéng,adjective|verb,NaN


In [8]:
df['hsk_level'] = df['hsk_level'].astype(str).replace({'7/09/2026':'7-9', '7-Sep': '7-9'})
df

,hsk_level,no_in_level,word,pinyin,part_of_speech,english
0,1,1,爱,ài,verb,to love
1,1,2,⼋,bā,number,eight
2,1,3,爸爸,bàba,noun,dad
3,1,4,吧,ba,auxiliary,"particle indicating suggestion, confirmation, ..."
4,1,5,⽩天,báitiān,noun,daytime
...,...,...,...,...,...,...
10995,7-9,5596,作息,zuòxī,verb,NaN
10996,7-9,5597,座右铭,zuòyòumíng,noun,NaN
10997,7-9,5598,作战,zuòzhàn,verb,NaN
10998,7-9,5599,做证,zuòzhèng,verb,NaN


In [9]:
df['hsk_level'].value_counts()

hsk_level
7-9    5600
6      1800
5      1600
4      1000
3       500
1       300
2       200
Name: count, dtype: int64

In [10]:
import unicodedata as ud

def normalize_nfkc(val):
    if pd.isna(val):
        return val
    return ud.normalize('NFKC', str(val))
    
for col in ['word', 'pinyin', 'english']:
    df[col] = df[col].apply(normalize_nfkc)

In [11]:
kou_mask = df['word'].str.contains('又', na = False) & df['pinyin'].str.contains('kǒu|kou', case = False, na = False)
print(f"Found {kou_mask.sum()} affected rows.")
display(df[kou_mask][['word', 'pinyin']].head())

Found 45 affected rows.


,word,pinyin
103,又,kǒu
400,门又,ménkǒu
762,路又,lùkǒu
1088,出又,chūkǒu
1439,又语,kǒuyǔ


In [12]:
df.loc[kou_mask, 'word'] = df.loc[kou_mask, 'word'].str.replace('又', '口')

In [13]:
import gzip
import urllib.request

local_gz_path = r"data/raw/cedict_1_0_ts_utf-8_mdbg.txt.gz"

line_pattern = re.compile(r"^(\S+)\s+(\S+)\s+\[(.*)\]\s+/(.*)/$")

cedict_dict = {}

with gzip.open(local_gz_path, "rt", encoding="utf-8") as f:
    for line in f:
        line_str = line.strip()
        if not line_str or line_str.startswith("#"):
            continue

        match = line_pattern.match(line_str)
        if match:
            _, simplified, _, definitions = match.groups()
            clean_defs = "; ".join(definitions.strip("/").split("/"))

            if simplified not in cedict_dict:
                cedict_dict[simplified] = clean_defs
print(f"Loaded {len(cedict_dict):,} entries from CC-CEDICT.")

Loaded 121,297 entries from CC-CEDICT.


In [14]:
missing_mask = df['english'].isna()
df.loc[missing_mask, 'english'] = df.loc[missing_mask, 'word'].map(cedict_dict)

print(f"Remaining missing definitions: {df['english'].isna().sum()}")

Remaining missing definitions: 23


In [15]:
df[df['english'].isna()][['word', 'hsk_level']]

,word,hsk_level
5893,成年2,7-9
6113,打2,7-9
6297,调2,7-9
6567,粉丝1,7-9
6568,粉丝2,7-9
7017,和2,7-9
7555,精彩纷呈,7-9
7792,口哨儿,7-9
8080,露2,7-9
8118,麻1,7-9


In [16]:
df['lookup_key'] = (
    df['word']
    .astype(str)
    .str.replace(r"\d+$", "", regex=True)
    .str.replace(r"儿$", "", regex=True)
    .str.strip()
)

In [17]:
df.loc[missing_mask, 'english'] = df.loc[missing_mask, 'lookup_key'].map(cedict_dict)
df.drop(columns=['lookup_key'], inplace=True)
print(f"Remaining missing definitions: {df['english'].isna().sum()}")

Remaining missing definitions: 2


In [18]:
final_fixes = {
    "精彩纷呈": "splendid; brilliant highlights presented one after another",
    "绕远儿": "to go out of one's way; to make a detour; to take the long way round",
}
df["english"] = df["english"].fillna(df["word"].map(final_fixes))
print(f"Remaining missing definitions: {df['english'].isna().sum()}") 

Remaining missing definitions: 0


In [19]:
df.to_csv("hsk_vocab_cleaned.csv", index=False, encoding="utf-8-sig")